# Stream Customers Data From Cloud Files to Delta Lake

1. Read files from cloud storage using DataStreamReader API
2. Transform the dataframe to add the following columns
    - file path: Cloud file path
    - ingestion date: Current Timestamp
3. Write the transformed data stream to Delta Lake Table

### 1. Read files using DataStreamReader API

- **Important**
    - Unlike **batch processing** where spark can infer schema automatically, in **Structured streaming**, schema inerence is disabled by default to improve performance and to avoid any inconsistencies when reading continuosly arriving data
    - We need to specify the shema

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType

customers_schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("customer_name", StringType()),
    StructField("date_of_birth", DateType()),
    StructField("telephone", StringType()),
    StructField("email", StringType()),
    StructField("member_since", DateType()),
    StructField("created_timestamp", TimestampType())
    ]
)
  

In [0]:
customers_df = (
        spark.readStream
        .format("json")
        .schema(customers_schema)
        .load("/Volumes/gizmobox/landing/operational_data/customers_stream/")
)

### 2. Transform the dataframe to add the following columns

- file path: Cloud file path
- ingestion date: Current Timestamp

In [0]:
from pyspark.sql.functions import current_timestamp, col
customers_transformed_df =(
    customers_df.withColumn("file_path", col("_metadata.file_path"))
                .withColumn("ingestion_date", current_timestamp())
)

### 3. Write the transformed data stream to Delta Table

In [0]:
streaming_query =(
    customers_transformed_df.writeStream 
    .format("delta")
    .option("checkpointLocation", "/Volumes/gizmobox/landing/operational_data/customers_stream/_checkpoint_stream")
    .toTable("gizmobox.bronze.customers_stream")
)

In [0]:
# para parar el streaming
streaming_query.stop()

In [0]:
%sql
select * from gizmobox.bronze.customers_stream

## Processing Modes
- **Micro-Batch Processing(default)**:spark collects the data from the source at smal intervals and processes that data as a batch, which is referred to as a micro batch. Once the microbatch completes, it again collects the data for the next batch and the cycle continues
- **Continuous Processing**: The data is processed record by recolrd with ultra low latency

## Control the streaming
Two important options that we can define on the write stream method to control the behavior of the microbatches
- Trigger: helps define the **frequency** of the microbatches
- Output Mode: controls **how** the process data is returned to the sink

#### Trigger
![imagen_1791159272345.png](./imagen_1791159272345.png "imagen_1791159272345.png")

#### OutputMode
![imagen_1791159378069.png](./imagen_1791159378069.png "imagen_1791159378069.png")

**Example**:

```
customers_transofmed_df.writeStream
                        .format("delta")
                        .outputMode("append")
                        .trigger(processingTime="2 minutes")
                        .option("checkpointLocation",
                                "/Volumes/gizmobox/landing/operational_data/customer_stream/_checkpoint")
                        .toTable("gizmobox.bronze.customers_streamed")

```

#### Checkpointing
![imagen_1791159572443.png](./imagen_1791159572443.png "imagen_1791159572443.png")

Checkpointing is a fault-tolerance mechanism that allows a query to recover from failures and resume processing from where it left off without data loss or duplication.

The location of the **CheckPoint** is used to stores metadata about streaming query, execution plan and also Tracks processed offsets and committed results

![imagen_1791159720463.png](./imagen_1791159720463.png "imagen_1791159720463.png")


When a micro-batch starts, the "read phase" arrives the offset in the checkpoint directory called the "offset log"

- **offset log** is a write ahead log, recording the offsets of the current batch before the execution of the batch.

- Then when the batch completes, the write process also writes the "commit log" with the offsets
- Write-Ahead Logs(WAL) and Checkpointing helps provide Fault Tolerance. **Offset log** and **Commit log** are used to provide **fault tolerance**. 

When a new batch starts Sparks first reads the offset log to get the start and the end offset of the previous batch

In order to guarantee that the data from the batcha was successfully processed, spark then reads the **commit log**. If batch id in the offset log matches with the commit log, then that indicates that the previous batch succeded.
![imagen_1791180927648.png](./imagen_1791180927648.png "imagen_1791180927648.png")

Idempotent Sinks enables exactly once guarantees

In case if a batch is processed again, the sink ignores the duplicates once, only some sinks are idempotent.
- Delta lake, kafka, etc.